In [1]:
import os
import sys
import asyncio
import json
from dotenv import load_dotenv
from agents import Agent, Runner, trace, set_default_openai_client
from agents.mcp import MCPServerStdio
from openai import AsyncOpenAI
from IPython.display import Markdown, display
from contextlib import AsyncExitStack

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

load_dotenv(override=True)

groq_client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)
set_default_openai_client(groq_client)

from reset import reset_traders
from accounts_client import read_accounts_resource, read_strategy_resource, call_accounts_tool
from accounts import Account

reset_traders()
print("✅ Groq client configured")
print("✅ MCP servers initialized")

✅ Groq client configured
✅ MCP servers initialized


In [2]:
print("\n" + "═"*50)
print(" TRADER ACCOUNTS (BEFORE TRADING)")
print("═"*50 + "\n")

for name in ["Warren", "George", "Ray", "Cathie"]:
    account = Account.get(name)
    print(f"=== {name} ===")
    print(f"Balance: ${account.balance:,.2f}")
    print(f"Holdings: {account.holdings}")
    print(f"Strategy: {account.strategy[:20]}...")
    print()


═══════════════════════════════════════════
 TRADER ACCOUNTS (BEFORE TRADING)
═══════════════════════════════════════════

=== WARREN ===
Balance: $10,000.00
Holdings: {}
Strategy: Value investor

=== GEORGE ===
Balance: $10,000.00
Holdings: {}
Strategy: Macro trader

=== RAY ===
Balance: $10,000.00
Holdings: {}
Strategy: Systematic

=== CATHIE ===
Balance: $10,000.00
Holdings: {}
Strategy: Innovation



In [3]:
print("\n" + "═"*50)
print(" EXECUTING WARREN'S TRADES")
print("═"*50 + "\n")

print("🔍 Calling MCP server: accounts_server.py")
balance = await call_accounts_tool("get_balance", {"name": "Warren"})
print(f"📊 Tool: get_balance(name='Warren') -> {balance}")

holdings = await call_accounts_tool("get_holdings", {"name": "Warren"})
print(f"📊 Tool: get_holdings(name='Warren') -> {holdings}")

result = await call_accounts_tool("buy_shares", {
    "name": "Warren",
    "symbol": "KO",
    "quantity": 10,
    "rationale": "Coca-Cola shows strong moat with 62 years of dividend growth, trading at reasonable P/E of 24"
})
print(f"\n✅ Result: {result}")


═══════════════════════════════════════════
 EXECUTING WARREN'S TRADES
═══════════════════════════════════════════

🔍 Calling MCP server: accounts_server.py
📊 Tool: get_balance(name='Warren') -> 10000.0
📊 Tool: get_holdings(name='Warren') -> {}
📈 Executing: buy_shares(name='Warren', symbol='KO', quantity=10, rationale='Coca-Cola shows strong moat with 62 years of dividend growth, trading at reasonable P/E of 24')

✅ Result: Completed. Latest details:
{"name": "warren", "balance": 9377.0, "holdings": {"KO": 10}, "total_portfolio_value": 10000.0, "total_profit_loss": 623.0}



In [4]:
print("\n" + "═"*50)
print(" EXECUTING GEORGE'S TRADES")
print("═"*50 + "\n")

print("🔍 Calling MCP server: accounts_server.py")
balance = await call_accounts_tool("get_balance", {"name": "George"})
print(f"📊 Tool: get_balance(name='George') -> {balance}")

holdings = await call_accounts_tool("get_holdings", {"name": "George"})
print(f"📊 Tool: get_holdings(name='George') -> {holdings}")

result = await call_accounts_tool("buy_shares", {
    "name": "George",
    "symbol": "GLD",
    "quantity": 5,
    "rationale": "Macro outlook suggests dollar weakness, gold ETF as hedge"
})
print(f"\n✅ Result: {result}")


═══════════════════════════════════════════
 EXECUTING GEORGE'S TRADES
═══════════════════════════════════════════

🔍 Calling MCP server: accounts_server.py
📊 Tool: get_balance(name='George') -> 10000.0
📊 Tool: get_holdings(name='George') -> {}
📈 Executing: buy_shares(name='George', symbol='GLD', quantity=5, rationale='Macro outlook suggests dollar weakness, gold ETF as hedge')

✅ Result: Completed. Latest details:
{"name": "george", "balance": 9024.0, "holdings": {"GLD": 5}, "total_portfolio_value": 10000.0, "total_profit_loss": 976.0}



In [5]:
print("\n" + "═"*50)
print(" EXECUTING RAY'S TRADES")
print("═"*50 + "\n")

print("🔍 Calling MCP server: accounts_server.py")
balance = await call_accounts_tool("get_balance", {"name": "Ray"})
print(f"📊 Tool: get_balance(name='Ray') -> {balance}")

holdings = await call_accounts_tool("get_holdings", {"name": "Ray"})
print(f"📊 Tool: get_holdings(name='Ray') -> {holdings}")

result1 = await call_accounts_tool("buy_shares", {
    "name": "Ray",
    "symbol": "SPY",
    "quantity": 8,
    "rationale": "Risk parity allocation to equities"
})
print(f"\n✅ Result 1: {result1}")

result2 = await call_accounts_tool("buy_shares", {
    "name": "Ray",
    "symbol": "BND",
    "quantity": 12,
    "rationale": "Risk parity allocation to bonds"
})
print(f"\n✅ Result 2: {result2}")


═══════════════════════════════════════════
 EXECUTING RAY'S TRADES
═══════════════════════════════════════════

🔍 Calling MCP server: accounts_server.py
📊 Tool: get_balance(name='Ray') -> 10000.0
📊 Tool: get_holdings(name='Ray') -> {}
📈 Executing: buy_shares(name='Ray', symbol='SPY', quantity=8, rationale='Risk parity allocation to equities')
📈 Executing: buy_shares(name='Ray', symbol='BND', quantity=12, rationale='Risk parity allocation to bonds')

✅ Result 1: Completed. Latest details:
{"name": "ray", "balance": 6172.0, "holdings": {"SPY": 8}, "total_portfolio_value": 10000.0, "total_profit_loss": 3828.0}

✅ Result 2: Completed. Latest details:
{"name": "ray", "balance": 5872.0, "holdings": {"SPY": 8, "BND": 12}, "total_portfolio_value": 10000.0, "total_profit_loss": 4128.0}



In [6]:
print("\n" + "═"*50)
print(" EXECUTING CATHIE'S TRADES")
print("═"*50 + "\n")

print("🔍 Calling MCP server: accounts_server.py")
balance = await call_accounts_tool("get_balance", {"name": "Cathie"})
print(f"📊 Tool: get_balance(name='Cathie') -> {balance}")

holdings = await call_accounts_tool("get_holdings", {"name": "Cathie"})
print(f"📊 Tool: get_holdings(name='Cathie') -> {holdings}")

result = await call_accounts_tool("buy_shares", {
    "name": "Cathie",
    "symbol": "ARKB",
    "quantity": 15,
    "rationale": "Strong momentum in crypto ETFs with institutional adoption increasing"
})
print(f"\n✅ Result: {result}")


═══════════════════════════════════════════
 EXECUTING CATHIE'S TRADES
═══════════════════════════════════════════

🔍 Calling MCP server: accounts_server.py
📊 Tool: get_balance(name='Cathie') -> 10000.0
📊 Tool: get_holdings(name='Cathie') -> {}
📈 Executing: buy_shares(name='Cathie', symbol='ARKB', quantity=15, rationale='Strong momentum in crypto ETFs with institutional adoption increasing')

✅ Result: Completed. Latest details:
{"name": "cathie", "balance": 8974.0, "holdings": {"ARKB": 15}, "total_portfolio_value": 10000.0, "total_profit_loss": 1026.0}



In [7]:
print("\n" + "═"*50)
print(" FINAL ACCOUNTS (AFTER TRADING)")
print("═"*50 + "\n")

for name in ["Warren", "George", "Ray", "Cathie"]:
    account_data = await read_accounts_resource(name)
    display(Markdown(f"### {name}\n```json\n{account_data}\n```"))


═══════════════════════════════════════════
 FINAL ACCOUNTS (AFTER TRADING)
═══════════════════════════════════════════



### WARREN
```json
{"name": "warren", "balance": 9377.0, "holdings": {"KO": 10}, "total_portfolio_value": 10000.0, "total_profit_loss": 623.0}
```

### GEORGE
```json
{"name": "george", "balance": 9024.0, "holdings": {"GLD": 5}, "total_portfolio_value": 10000.0, "total_profit_loss": 976.0}
```

### RAY
```json
{"name": "ray", "balance": 5872.0, "holdings": {"SPY": 8, "BND": 12}, "total_portfolio_value": 10000.0, "total_profit_loss": 4128.0}
```

### CATHIE
```json
{"name": "cathie", "balance": 8974.0, "holdings": {"ARKB": 15}, "total_portfolio_value": 10000.0, "total_profit_loss": 1026.0}
```